# NB3 — Hyperparameter Sensitivity Heatmap + Pruning/Quantization

Addresses:
- **R4 pt.3**: Hyperparameter sensitivity grid (gate entropy λ × label smoothing), reported as an accuracy heatmap (reduced scale for the sweep)
- **R4 pt.4**: Magnitude-based pruning (50% sparsity) + post-training INT8 quantization, with accuracy/size/latency trade-offs reported using the best hyperparameters from the sweep

Run on Kaggle T4 (can run independently of NB1/NB2). Exports `nb3_export.pkl`.

In [ ]:
!pip install EMD-signal

In [ ]:
import os
OUTPUT_DIR = '/kaggle/working/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Figures will be saved to: {os.path.abspath(OUTPUT_DIR)}')

# -*- coding: utf-8 -*-
"""
NB3 — Hyperparameter Sensitivity Heatmap + Pruning/Quantization
Addresses:
  - R4 pt.3 / Editor: hyperparameter sensitivity analysis (gate entropy
    lambda x label smoothing grid), reported as a heatmap of mean accuracy
  - R4 pt.4: post-training pruning and INT8 quantization, with accuracy
    and model-size/latency trade-off reported (deployment constraints)
Scale: reduced (N_SENS, fewer epochs per grid cell) to fit a 12hr budget —
this is a SENSITIVITY/TREND study, not a final-accuracy claim.
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from scipy.signal import hilbert, butter, lfilter
from scipy.stats import skew, kurtosis, entropy
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import time, gc, logging, warnings, pickle
logging.getLogger('PyEMD').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU ready: {len(gpus)} device(s), mixed_float16 enabled")

# ── Scale (reduced for sensitivity grid — many runs needed) ────────────────
N_SENS      = 400          # samples/class/SNR — smaller for grid sweep
GASF_SENS   = 64
SEQ_SENS    = 128
BATCH_SENS  = 128
EPOCHS_SENS = 30
PATIENCE_SENS = 8

# Final model for pruning/quant uses moderate scale (fits in time budget
# alongside the sensitivity grid)
N_FINAL     = 1000
GASF_FINAL  = 96
SEQ_FINAL   = 256
BATCH_FINAL = 64
EPOCHS_FINAL= 60
PATIENCE_FINAL = 15

SNRs_to_test = [-10, -5, 0, 5, 10, 20]

# Hyperparameter grid (R4 pt.3)
GATE_LAMBDAS    = [0.0, 0.02, 0.05, 0.1, 0.2]
LABEL_SMOOTHINGS = [0.0, 0.05, 0.1, 0.15, 0.2]

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 120, 'savefig.dpi': 200,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
})

def butter_lowpass(cutoff, fs, order=9):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return b, a

# ══════════════════════════════════════════════════════════════════════════
# DATA GENERATION + FEATURES
# ══════════════════════════════════════════════════════════════════════════
def generate_signals(n_per_class, snrs, num_samples=1024):
    fs, fc = 100000, 5000
    t = np.arange(num_samples) / fs
    X, Y, SNR_labels = [], [], []
    for snr in snrs:
        for _ in range(n_per_class):
            noise    = np.random.randn(num_samples)
            b, a     = butter_lowpass(1000, fs, order=9)
            m_t      = lfilter(b, a, noise)
            m_t_norm = m_t / np.max(np.abs(m_t))
            am   = (1 + 0.8*m_t_norm)*np.cos(2*np.pi*fc*t)
            pm   = np.cos(2*np.pi*fc*t + np.pi*m_t_norm)
            fm   = np.cos(2*np.pi*fc*t + 2*np.pi*1000*np.cumsum(m_t)/fs)
            m_hat = np.imag(hilbert(m_t))
            ssb  = m_t*np.cos(2*np.pi*fc*t) - m_hat*np.sin(2*np.pi*fc*t)
            for idx, sig in enumerate([am, pm, fm, ssb]):
                sp  = np.mean(sig**2)
                np_ = sp / (10**(snr/10))
                ns  = sig + np.sqrt(np_)*np.random.randn(num_samples)
                X.append((ns - ns.mean()) / ns.std())
                Y.append(idx)
                SNR_labels.append(snr)
    return (np.array(X, dtype=np.float32), np.array(Y, dtype=np.int8),
            np.array(SNR_labels, dtype=np.int8))

def extract_emd_features(signal):
    emd  = EMD()
    imfs = emd.emd(signal, max_imf=3)
    features = []
    for i in range(3):
        if i < len(imfs):
            imf   = imfs[i]
            power = np.abs(imf)**2
            prob  = power / (np.sum(power) + 1e-10)
            features.extend([np.std(imf), skew(imf), kurtosis(imf),
                              np.sum(power), entropy(prob)])
        else:
            features.extend([0.0]*5)
    return np.array(features, dtype=np.float32)

def compute_gasf(signal, size):
    sr     = np.interp(np.linspace(0, len(signal), size), np.arange(len(signal)), signal)
    lo, hi = sr.min(), sr.max()
    n      = (sr - lo) / (hi - lo + 1e-8) * 2 - 1
    phi    = np.arccos(np.clip(n, -1, 1))
    c, s   = np.cos(phi), np.sin(phi)
    return (np.outer(c, c) - np.outer(s, s)).astype(np.float32)

def extract_sequence(signal, seq_len):
    stride = max(1, len(signal)//seq_len)
    seq = signal[::stride][:seq_len]
    if len(seq) < seq_len:
        seq = np.pad(seq, (0, seq_len-len(seq)))
    return seq.astype(np.float32).reshape(seq_len,1)

def extract_all(X_raw_arr, gasf_size, seq_len, tag=""):
    emd_l, gaf_l, seq_l = [], [], []
    for i, sig in enumerate(X_raw_arr):
        if i % 5000 == 0 and i > 0:
            print(f"  [{tag}] {i}/{len(X_raw_arr)}...")
        emd_l.append(extract_emd_features(sig))
        gaf_l.append(compute_gasf(sig, gasf_size))
        seq_l.append(extract_sequence(sig, seq_len))
    return (np.array(emd_l,dtype=np.float32).reshape(-1,15,1),
            np.array(gaf_l,dtype=np.float32).reshape(-1,gasf_size,gasf_size,1),
            np.array(seq_l,dtype=np.float32))

# ── Sensitivity-grid dataset (smaller) ─────────────────────────────────────
print("Generating sensitivity-grid dataset...")
X_raw_s, Y_s, SNR_s = generate_signals(N_SENS, SNRs_to_test)
X_emd_s, X_gaf_s, X_seq_s = extract_all(X_raw_s, GASF_SENS, SEQ_SENS, tag="sens")
del X_raw_s; gc.collect()
Y_oh_s = to_categorical(Y_s, num_classes=4)
tr_s, te_s = train_test_split(np.arange(len(Y_s)), test_size=0.2, random_state=42, stratify=Y_s)
train_sens = {"emd_input": X_emd_s[tr_s], "gaf_input": X_gaf_s[tr_s], "seq_input": X_seq_s[tr_s]}
test_sens  = {"emd_input": X_emd_s[te_s], "gaf_input": X_gaf_s[te_s], "seq_input": X_seq_s[te_s]}
y_train_sens = Y_oh_s[tr_s]; y_test_sens = Y_oh_s[te_s]; snr_test_sens = SNR_s[te_s]
del X_emd_s, X_gaf_s, X_seq_s, Y_s, Y_oh_s; gc.collect()
print(f"Sensitivity dataset ready: train={len(tr_s)}, test={len(te_s)}")

# ── Final dataset (for pruning/quant — moderate scale) ─────────────────────
print("\nGenerating final-model dataset (for pruning/quantization)...")
X_raw_f, Y_f, SNR_f = generate_signals(N_FINAL, SNRs_to_test)
X_emd_f, X_gaf_f, X_seq_f = extract_all(X_raw_f, GASF_FINAL, SEQ_FINAL, tag="final")
del X_raw_f; gc.collect()
Y_oh_f = to_categorical(Y_f, num_classes=4)
tr_f, te_f = train_test_split(np.arange(len(Y_f)), test_size=0.2, random_state=42, stratify=Y_f)
train_final = {"emd_input": X_emd_f[tr_f], "gaf_input": X_gaf_f[tr_f], "seq_input": X_seq_f[tr_f]}
test_final  = {"emd_input": X_emd_f[te_f], "gaf_input": X_gaf_f[te_f], "seq_input": X_seq_f[te_f]}
y_train_final = Y_oh_f[tr_f]; y_test_final = Y_oh_f[te_f]; snr_test_final = SNR_f[te_f]
del X_emd_f, X_gaf_f, X_seq_f, Y_f, Y_oh_f; gc.collect()
print(f"Final dataset ready: train={len(tr_f)}, test={len(te_f)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL BUILDER (parameterized by gate entropy lambda, label smoothing, scale)
# ══════════════════════════════════════════════════════════════════════════
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1,1,filters))(se)
    se = layers.Dense(max(filters//ratio,1), activation='relu',
                       kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid',
                       kernel_initializer='he_normal', use_bias=False)(se)
    return layers.Multiply()([x, se])

def gaf_cnn_branch(gaf_input, full_res):
    x = layers.Concatenate()([
        layers.Conv2D(16,(3,3),padding='same',activation='relu')(gaf_input),
        layers.Conv2D(16,(5,5),padding='same',activation='relu')(gaf_input),
    ])
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    x = layers.Conv2D(64,(3,3),padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    nf = 128 if full_res else 64
    x = layers.Conv2D(nf,(3,3),padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    return layers.Concatenate()([layers.GlobalAveragePooling2D()(x),
                                  layers.GlobalMaxPooling2D()(x)])

class EntropyRegGateLayer(layers.Layer):
    def __init__(self, lam=0.05, **kwargs):
        super().__init__(**kwargs)
        self.lam = lam
        self.dense = layers.Dense(3, activation='softmax', dtype='float32')
    def call(self, inputs):
        gate = self.dense(inputs)
        if self.lam > 0:
            mean_g = tf.reduce_mean(gate, axis=0, keepdims=True)
            var_g  = tf.reduce_mean(tf.square(gate-mean_g), axis=0)
            self.add_loss(self.lam * tf.exp(-10.0*tf.reduce_mean(var_g)))
        return gate

def build_model(gasf_size, seq_len, gate_lambda=0.05, label_smoothing=0.05, lr=0.0003):
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(gasf_size,gasf_size,1), name="gaf_input")
    seq_input = layers.Input(shape=(seq_len,1), name="seq_input")

    emd_raw = layers.Flatten()(emd_input)
    x1 = layers.Dense(64, activation='relu')(emd_raw)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Dense(32, activation='relu')(x1)

    x2 = gaf_cnn_branch(gaf_input, full_res=(gasf_size>=96))

    x3 = layers.Conv1D(32,7,strides=2,padding='same',activation='relu')(seq_input)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Conv1D(64,5,strides=2,padding='same',activation='relu')(x3)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Conv1D(64,3,strides=1,padding='same',activation='relu')(x3)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Bidirectional(layers.GRU(64,return_sequences=True,dropout=0.1))(x3)
    x3 = layers.Bidirectional(layers.GRU(32,return_sequences=False,dropout=0.1))(x3)

    concat_all = layers.Concatenate()([x1,x2,x3,emd_raw])
    gate_hidden = layers.Dense(128, activation='relu')(concat_all)
    gate = EntropyRegGateLayer(lam=gate_lambda)(gate_hidden)

    x1_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,0],1))([x1,gate])
    x2_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,1],1))([x2,gate])
    x3_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,2],1))([x3,gate])

    merged = layers.Concatenate()([x1_g,x2_g,x3_g,emd_raw])
    z = layers.Dense(256, activation='relu')(merged)
    z = layers.BatchNormalization()(z)
    z = layers.Dropout(0.25)(z)
    z = layers.Dense(128, activation='relu')(z)
    z = layers.Dropout(0.15)(z)
    z = layers.Dense(64, activation='relu')(z)
    out = layers.Dense(4, activation='softmax', dtype='float32')(z)

    model = Model(inputs=[emd_input,gaf_input,seq_input], outputs=out)
    loss = (tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)
            if label_smoothing > 0 else 'categorical_crossentropy')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                   loss=loss, metrics=['accuracy'])
    return model

print("NB3 model builder ready.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# HYPERPARAMETER SENSITIVITY SWEEP (R4 pt.3)
# Grid over gate_entropy_lambda x label_smoothing, reduced scale,
# 1 run per cell, EPOCHS_SENS max epochs with early stopping.
# Reports mean test accuracy across SNRs for each grid cell.
# ══════════════════════════════════════════════════════════════════════════
sensitivity_grid = np.zeros((len(GATE_LAMBDAS), len(LABEL_SMOOTHINGS)))
sensitivity_detail = {}

total_runs = len(GATE_LAMBDAS) * len(LABEL_SMOOTHINGS)
run_i = 0
t_grid0 = time.time()

for gi, glam in enumerate(GATE_LAMBDAS):
    for li, lsm in enumerate(LABEL_SMOOTHINGS):
        run_i += 1
        print(f"\n[{run_i}/{total_runs}] gate_lambda={glam}, label_smoothing={lsm}")
        tf.random.set_seed(123); np.random.seed(123)
        m = build_model(GASF_SENS, SEQ_SENS, gate_lambda=glam, label_smoothing=lsm)
        h = m.fit(train_sens, y_train_sens, validation_data=(test_sens, y_test_sens),
                  batch_size=BATCH_SENS, epochs=EPOCHS_SENS,
                  callbacks=[EarlyStopping('val_loss', patience=PATIENCE_SENS,
                                            restore_best_weights=True)],
                  verbose=0)
        pr = m.predict(test_sens, verbose=0)
        acc = np.mean(np.argmax(pr,1)==np.argmax(y_test_sens,1)) * 100
        sensitivity_grid[gi, li] = acc
        sensitivity_detail[(glam,lsm)] = {
            'acc': acc, 'epochs': len(h.history['loss']),
            'val_acc_final': h.history['val_accuracy'][-1]*100
        }
        print(f"  -> test acc = {acc:.2f}%  (epochs={len(h.history['loss'])})")
        tf.keras.backend.clear_session(); gc.collect()

print(f"\nSensitivity grid complete. Total time: {(time.time()-t_grid0)/60:.1f} min")
print("\nGrid (rows=gate_lambda, cols=label_smoothing):")
print("           " + "".join([f"{l:>8}" for l in LABEL_SMOOTHINGS]))
for gi, glam in enumerate(GATE_LAMBDAS):
    print(f"  λ={glam:<6}" + "".join([f"{sensitivity_grid[gi,li]:>8.2f}" for li in range(len(LABEL_SMOOTHINGS))]))

best_idx = np.unravel_index(np.argmax(sensitivity_grid), sensitivity_grid.shape)
best_lambda = GATE_LAMBDAS[best_idx[0]]
best_lsm    = LABEL_SMOOTHINGS[best_idx[1]]
print(f"\nBest combination: gate_lambda={best_lambda}, label_smoothing={best_lsm} "
      f"-> {sensitivity_grid[best_idx]:.2f}%")

# ── Heatmap plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(sensitivity_grid, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=LABEL_SMOOTHINGS, yticklabels=GATE_LAMBDAS,
            cbar_kws={'label':'Test Accuracy (%)'}, ax=ax)
ax.set_xlabel('Label Smoothing')
ax.set_ylabel('Gate Entropy λ')
ax.set_title('Hyperparameter Sensitivity: Test Accuracy (%)\n'
              f'(N={N_SENS}/class/SNR, reduced scale, R4 pt.3)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p40_hyperparam_sensitivity_heatmap.png'))
plt.show()
print("Saved: p40_hyperparam_sensitivity_heatmap.png")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PRUNING + QUANTIZATION (R4 pt.4)
# 1. Train a final model at moderate scale (best hyperparams from sweep)
# 2. Apply magnitude-based pruning (tensorflow_model_optimization)
# 3. Apply post-training INT8 quantization (TFLite)
# 4. Report accuracy, model size, and inference latency for: baseline,
#    pruned, quantized
# ══════════════════════════════════════════════════════════════════════════
print(f"\nTraining final model with best hyperparams from sweep: "
      f"gate_lambda={best_lambda}, label_smoothing={best_lsm}")
tf.random.set_seed(42); np.random.seed(42)
m_final = build_model(GASF_FINAL, SEQ_FINAL, gate_lambda=best_lambda,
                       label_smoothing=best_lsm)
h = m_final.fit(train_final, y_train_final, validation_data=(test_final, y_test_final),
                batch_size=BATCH_FINAL, epochs=EPOCHS_FINAL,
                callbacks=[EarlyStopping('val_loss', patience=PATIENCE_FINAL,
                                          restore_best_weights=True)],
                verbose=1)
pr = m_final.predict(test_final, verbose=0)
baseline_acc = np.mean(np.argmax(pr,1)==np.argmax(y_test_final,1))*100
print(f"\nFinal model trained. Test accuracy = {baseline_acc:.2f}%, "
      f"params = {m_final.count_params():,}")

m_final.save('/kaggle/working/model_final_fp32.keras')
baseline_size_mb = os.path.getsize('/kaggle/working/model_final_fp32.keras') / (1024*1024)

# ══════════════════════════════════════════════════════════════════════════
# PRUNING
# ══════════════════════════════════════════════════════════════════════════
try:
    import tensorflow_model_optimization as tfmot
    TFMOT_AVAILABLE = True
except ImportError:
    print("Installing tensorflow_model_optimization...")
    os.system("pip install -q tensorflow_model_optimization --break-system-packages")
    import tensorflow_model_optimization as tfmot
    TFMOT_AVAILABLE = True

if TFMOT_AVAILABLE:
    prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude
    PRUNE_EPOCHS = min(15, EPOCHS_FINAL // 2)
    pruning_params = {
        'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
            initial_sparsity=0.0, final_sparsity=0.5,
            begin_step=0, end_step=int(len(train_final['emd_input'])/BATCH_FINAL)*PRUNE_EPOCHS)
    }
    m_prune = prune_low_magnitude(
        tf.keras.models.load_model('/kaggle/working/model_final_fp32.keras'),
        **pruning_params)
    loss = (tf.keras.losses.CategoricalCrossentropy(label_smoothing=best_lsm)
            if best_lsm > 0 else 'categorical_crossentropy')
    m_prune.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=loss, metrics=['accuracy'])

    print(f"\nFine-tuning pruned model ({PRUNE_EPOCHS} epochs, target sparsity=50%)...")
    m_prune.fit(train_final, y_train_final, validation_data=(test_final, y_test_final),
                batch_size=BATCH_FINAL, epochs=PRUNE_EPOCHS,
                callbacks=[tfmot.sparsity.keras.UpdatePruningStep()], verbose=1)

    m_prune_export = tfmot.sparsity.keras.strip_pruning(m_prune)
    pr = m_prune_export.predict(test_final, verbose=0)
    pruned_acc = np.mean(np.argmax(pr,1)==np.argmax(y_test_final,1))*100

    m_prune_export.save('/kaggle/working/model_pruned.keras')
    pruned_size_mb = os.path.getsize('/kaggle/working/model_pruned.keras') / (1024*1024)
    print(f"Pruned model: acc={pruned_acc:.2f}%, size={pruned_size_mb:.2f}MB "
          f"(target 50% sparsity)")
else:
    pruned_acc, pruned_size_mb = np.nan, np.nan
    m_prune_export = m_final

# ══════════════════════════════════════════════════════════════════════════
# POST-TRAINING INT8 QUANTIZATION (TFLite)
# ══════════════════════════════════════════════════════════════════════════
print("\nApplying post-training INT8 quantization...")

def representative_dataset():
    n = min(200, len(train_final['emd_input']))
    for i in range(n):
        yield [train_final['emd_input'][i:i+1].astype(np.float32),
               train_final['gaf_input'][i:i+1].astype(np.float32),
               train_final['seq_input'][i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(m_prune_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
                                         tf.lite.OpsSet.TFLITE_BUILTINS]
try:
    tflite_model = converter.convert()
    with open('/kaggle/working/model_quantized.tflite', 'wb') as f:
        f.write(tflite_model)
    quant_size_mb = os.path.getsize('/kaggle/working/model_quantized.tflite') / (1024*1024)

    # Evaluate TFLite model accuracy
    interpreter = tf.lite.Interpreter(model_path='/kaggle/working/model_quantized.tflite')
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    name_to_idx = {d['name'].split(':')[0].split('_int8')[0]: d['index'] for d in input_details}
    # Fallback: map by order if names don't match cleanly
    n_test = min(500, len(test_final['emd_input']))  # subset for speed
    correct = 0
    lat_total = 0.0
    for i in range(n_test):
        for d in input_details:
            key = None
            for k in ['emd_input','gaf_input','seq_input']:
                if k in d['name']:
                    key = k; break
            if key is None:
                # assume order emd, gaf, seq
                idx_pos = input_details.index(d)
                key = ['emd_input','gaf_input','seq_input'][idx_pos]
            val = test_final[key][i:i+1]
            if d['dtype'] == np.int8:
                scale, zp = d['quantization']
                val = (val/scale + zp).astype(np.int8)
            interpreter.set_tensor(d['index'], val.astype(d['dtype']))
        t0 = time.time()
        interpreter.invoke()
        lat_total += (time.time()-t0)
        out = interpreter.get_tensor(output_details[0]['index'])
        if output_details[0]['dtype'] == np.int8:
            scale, zp = output_details[0]['quantization']
            out = (out.astype(np.float32)-zp)*scale
        pred = np.argmax(out, axis=1)[0]
        true = np.argmax(y_test_final[i])
        correct += int(pred==true)
    quant_acc = correct/n_test*100
    quant_latency_ms = (lat_total/n_test)*1000
    print(f"Quantized model: acc={quant_acc:.2f}% (n={n_test}), "
          f"size={quant_size_mb:.3f}MB, latency={quant_latency_ms:.3f}ms/sample")
except Exception as e:
    print(f"Quantization conversion/eval failed: {e}")
    quant_acc, quant_size_mb, quant_latency_ms = np.nan, np.nan, np.nan

# ── Baseline (FP32) latency ─────────────────────────────────────────────────
n_lat = min(200, len(test_final['emd_input']))
t0 = time.time()
for i in range(n_lat):
    _ = m_final.predict({k:v[i:i+1] for k,v in test_final.items()}, verbose=0)
fp32_latency_ms = (time.time()-t0)/n_lat*1000

print("\n" + "═"*70)
print("PRUNING + QUANTIZATION SUMMARY  (R4 pt.4)")
print("═"*70)
print(f"{'Variant':<22}{'Accuracy':>10}{'Size (MB)':>12}{'Latency (ms)':>14}")
print("-"*70)
print(f"{'Baseline FP32':<22}{baseline_acc:>9.2f}%{baseline_size_mb:>12.3f}{fp32_latency_ms:>14.3f}")
print(f"{'Pruned (50% sparse)':<22}{pruned_acc:>9.2f}%{pruned_size_mb:>12.3f}{'(same as FP32)':>14}")
print(f"{'Pruned + INT8 quant':<22}{quant_acc:>9.2f}%{quant_size_mb:>12.3f}{quant_latency_ms:>14.3f}")
print("═"*70)
compression = baseline_size_mb / quant_size_mb if quant_size_mb and not np.isnan(quant_size_mb) else np.nan
print(f"\nCompression ratio (FP32 -> INT8): {compression:.2f}x")
print(f"Accuracy delta (FP32 -> pruned+INT8): {quant_acc-baseline_acc:+.2f}pp")

# ── Plot ─────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5.5))
variants = ['FP32\n(baseline)', 'Pruned\n(50% sparse)', 'Pruned+INT8\n(quantized)']
accs_plot = [baseline_acc, pruned_acc, quant_acc]
sizes_plot = [baseline_size_mb, pruned_size_mb, quant_size_mb]
colors = ['#2E4057','#48CAE4','#E84855']

ax1.bar(variants, accs_plot, color=colors, alpha=0.85)
for i,a in enumerate(accs_plot):
    if not np.isnan(a):
        ax1.text(i, a+0.5, f'{a:.2f}%', ha='center', fontweight='bold')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('(a) Accuracy', fontweight='bold')
ax1.set_ylim(0,105)
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)

ax2.bar(variants, sizes_plot, color=colors, alpha=0.85)
for i,s in enumerate(sizes_plot):
    if not np.isnan(s):
        ax2.text(i, s+max(sizes_plot)*0.02, f'{s:.3f}MB', ha='center', fontweight='bold')
ax2.set_ylabel('Model Size (MB)')
ax2.set_title('(b) Model Size', fontweight='bold')
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Pruning + Quantization Trade-offs (R4 pt.4)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p41_pruning_quantization.png'))
plt.show()
print("Saved: p41_pruning_quantization.png")

# ── Export ──────────────────────────────────────────────────────────────
nb3_export = {
    'sensitivity_grid': sensitivity_grid,
    'sensitivity_detail': sensitivity_detail,
    'GATE_LAMBDAS': GATE_LAMBDAS,
    'LABEL_SMOOTHINGS': LABEL_SMOOTHINGS,
    'best_lambda': best_lambda,
    'best_lsm': best_lsm,
    'baseline_acc': baseline_acc, 'baseline_size_mb': baseline_size_mb,
    'pruned_acc': pruned_acc, 'pruned_size_mb': pruned_size_mb,
    'quant_acc': quant_acc, 'quant_size_mb': quant_size_mb,
    'fp32_latency_ms': fp32_latency_ms, 'quant_latency_ms': quant_latency_ms,
}
with open('/kaggle/working/nb3_export.pkl','wb') as f:
    pickle.dump(nb3_export, f)
print("\nSaved nb3_export.pkl")

import glob
pngs = glob.glob(os.path.join(OUTPUT_DIR,'*.png'))
if pngs:
    os.system(f'zip -r /kaggle/working/all_figures_nb3.zip {OUTPUT_DIR}/')
    print(f'Zipped {len(pngs)} figures → /kaggle/working/all_figures_nb3.zip')
